In [77]:
import pandas as pd
import os
import yaml
from pathlib import Path
import sys
import importlib

%load_ext autoreload
%autoreload 2

PROJECT_ROOT = Path.cwd().parent
import functions as fn
with open(PROJECT_ROOT / "config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

#project_root = Path.cwd().parent
#sys.path.append(str(project_root))

             
#Data raw folder path:
raw_folder = r"C:\Users\ziden\Desktop\Trainings\RNCP-Project\data\raw"
#Data clean folder path:
clean_folder = r"C:\Users\ziden\Desktop\Trainings\RNCP-Project\data\clean"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [79]:
importlib.reload(fn)

<module 'functions' from 'C:\\Users\\ziden\\Desktop\\Trainings\\RNCP-Project\\functions.py'>

In [37]:
#-----------------------------------------------------------------------------
# 1. JOINING Population data with geo data: pop_geo_merge_df(population_df, geo_df)
#-----------------------------------------------------------------------------
#open Geographic file

geo_df_file_path = os.path.join(clean_folder, "insee_geo.csv")
geo_df = pd.read_csv(geo_df_file_path, sep=",", dtype={"insee_com": "str"}, encoding="latin1")

#open population file
population_df_file_path = os.path.join(clean_folder, "insee_com_population.csv")
population_df = pd.read_csv(population_df_file_path, sep=",", dtype={"insee_com": "str"}, encoding="latin1")

pop_geo_clean_df = fn.pop_geo_merge_df(population_df, geo_df)
pop_geo_clean_df.head()



File 'geo_population_table.csv' is successfully saved to 'data/clean' folder.


,insee_com,year,measure,population,com_name,insee_dep,dep_name,insee_reg,reg_name
0,01259,2023,PMUN,709.0,MONTCET,01,AIN,84,Auvergne-Rhône-Alpes
1,01216,2023,PMUN,886.0,LHUIS,01,AIN,84,Auvergne-Rhône-Alpes
2,01124,2023,PMUN,703.0,CORMOZ,01,AIN,84,Auvergne-Rhône-Alpes
3,01196,2023,PMUN,1261.0,JAYAT,01,AIN,84,Auvergne-Rhône-Alpes
4,02073,2023,PMUN,722.0,BERRY AU BAC,02,AISNE,32,Hauts-de-France


In [105]:
#-----------------------------------------------------------------------------
# 2. CLEANING ARCEP QOS FILE. function: clean_qos_df(raw_qos_df): "file1" in data/raw folder
#-----------------------------------------------------------------------------

clean_qos_df= fn.clean_qos_df("file1")
print(clean_qos_df.dtypes)
clean_qos_df.insee_com.unique()

File '5G_qos_clean.csv' is successfully saved to 'data/clean' folder.
measure_id                             int64
acess_duration                       float64
bitrate_dl                           float64
bitrate_ul                           float64
date_start                    datetime64[us]
hour_start                            object
insee_com                             string
latitude_start                       float64
loaded_in_less_10_secondes           float64
loaded_in_less_5_secondes            float64
longitude_start                      float64
operator                                 str
quality_correct                      float64
quality_perfect                      float64
rsrp                                 float64
rsrq                                 float64
protocole                                str
situation                                str
techno_start                             str
terminal                                 str
url                           

<StringArray>
['40088', '40279', '40202', '40207', '40312', '64102', '64122', '64125',
 '64024', '64558',
 ...
 '26206', '26307', '01383', '01443', '01389', '69102', '03118', '03023',
 '03310', '03264']
Length: 1486, dtype: string

In [106]:
communes_with_measurements = clean_qos_df.insee_com.unique()
len(communes_with_measurements)

1486

In [107]:
#Determine the list of communes to keep: should be same in Qos_df and in geo_pop files
communes_with_measurements = clean_qos_df.insee_com.unique()

communes_in_geo_po_file = pop_geo_clean_df.insee_com.unique()
print("Number of communes in qos_file is:", len(communes_with_measurements))
print("Number of communes in geo_pop_file:", len(communes_in_geo_po_file))

common_communes_geo_qos = set(communes_with_measurements) & set(communes_in_geo_po_file)
print("Number of communes present in both files: 'pop_geo_clean_df' and 'clean_qos_df' is:", len(common_communes_geo_qos))

Number of communes in qos_file is: 1486
Number of communes in geo_pop_file: 34858
Number of communes present in both files: 'pop_geo_clean_df' and 'clean_qos_df' is: 1486


In [108]:
#-----------------------------------------------------------------------------
# 3. CLEANING INSEE SITES FILE. file2 in "data/raw" folder
#-----------------------------------------------------------------------------

sites_clean_df = fn.clean_sites_file("file2")
sites_clean_df.columns

File 'insee_sites_clean.csv' is successfully saved to 'data/clean' folder.


Index(['site_id', 'code_op', 'nom_op', 'num_site', 'id_site_partage',
       'id_station_anfr', 'latitude', 'longitude', 'nom_reg', 'nom_dep',
       'insee_dep', 'nom_com', 'insee_com', 'site_4g', 'site_5g',
       'mes_4g_trim', 'date_ouverturecommerciale_5g', 'site_5g_700_m_hz',
       'site_5g_800_m_hz', 'site_5g_1800_m_hz', 'site_5g_2100_m_hz',
       'site_5g_3500_m_hz', 'operator_id'],
      dtype='str')

In [109]:
sites_clean_df.insee_com.unique()

<StringArray>
['67309', '85226', '13055', '93066', '62447', '25223', '33063', '51507',
 '21231', '59178',
 ...
 '53197', '56065', '59193', '65296', '66174', '76752', '77172', '31515',
 '88524', '94048']
Length: 1486, dtype: str

In [104]:
communes_in_insee_sites_df = sites_clean_df.insee_com.unique()
print("Number of total communes in sites_clean_df is:", len(communes_in_insee_sites_df))
print("Number of communes in qos_file is:", len(communes_with_measurements))
print("Number of communes in geo_pop_file:", len(communes_in_geo_po_file))

#Determine the communes present in the 3 files: qos_df_clean, geo_pop_clean_df and sites_clean_df
test_communes = set(communes_with_measurements) & set(communes_in_geo_po_file) & set(communes_in_insee_sites_df)
print("Number of communes in files: qos_df_clean, geo_pop_clean_df and sites_clean_df is:", len(test_communes))

#print("\nTest communes are:", test_communes)
#Save the list of test communes in data_clean folder
test_communes_series = pd.Series(list(test_communes))
test_communes_series.to_csv(os.path.join(clean_folder, "test_communes.csv"), index=False, encoding="utf-8")
print("file 'test_communes.csv' is successfully saved in 'data/clean' folder")

test_communes_series

Number of total communes in sites_clean_df is: 20690
Number of communes in qos_file is: 1610
Number of communes in geo_pop_file: 34858
Number of communes in files: qos_df_clean, geo_pop_clean_df and sites_clean_df is: 1486
file 'test_communes.csv' is successfully saved in 'data/clean' folder


0       79137
1       07324
2       53201
3       17172
4       62548
        ...  
1481    62126
1482    89344
1483    40001
1484    09282
1485    78551
Length: 1486, dtype: str